# AeroInspect — YOLOv8 Aircraft Defect Detection Training
**Platform:** Kaggle GPU (T4)
**Model:** YOLOv8m (medium — best balance of speed vs accuracy)
**Target:** Detect cracks, corrosion, dents, surface damage on aircraft components

In [ ]:
# Step 1 — Install dependencies
!pip install ultralytics roboflow --quiet

In [ ]:
import os
from roboflow import Roboflow
from ultralytics import YOLO
import yaml
import shutil
from pathlib import Path

ROBOFLOW_API_KEY = 'dIQkP88HGQBVhOtvhXg1'
WORKSPACE_ID = 'keerthishrees-workspace'
WORK_DIR = '/kaggle/working'
DATA_DIR = f'{WORK_DIR}/datasets'

## Step 2 — Download Datasets from Roboflow

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def download_dataset(workspace, project_name, dest, label):
    try:
        project = rf.workspace(workspace).project(project_name)
        versions = project.versions()
        latest = versions[-1].version
        print(f'  Found {len(versions)} version(s), using version {latest}')
        dataset = project.version(latest).download('yolov8', location=dest)
        print(f'✅ {label} downloaded')
        return True
    except Exception as e:
        print(f'❌ {label} failed: {e}')
        return False

print('Downloading Dataset 1...')
download_dataset('university-of-technology-sydney-21uto', 'aircraft-defect-detection', f'{DATA_DIR}/ds1', 'Dataset 1')

print('Downloading Dataset 2...')
download_dataset('lemi-debele', 'aircraft-surface-damage', f'{DATA_DIR}/ds2', 'Dataset 2')

print('Downloading Dataset 3...')
download_dataset('youssef-donia-fhktl', 'aircraft-damage-detection-2', f'{DATA_DIR}/ds3', 'Dataset 3')

print('\nDownloaded datasets:', os.listdir(DATA_DIR))

## Step 3 — Merge Datasets

In [ ]:
MERGED_DIR = f'{WORK_DIR}/merged'

for split in ['train', 'valid', 'test']:
    os.makedirs(f'{MERGED_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{MERGED_DIR}/{split}/labels', exist_ok=True)

def merge_dataset(src_dir, split, prefix):
    img_src = Path(src_dir) / split / 'images'
    lbl_src = Path(src_dir) / split / 'labels'
    if not img_src.exists():
        return 0
    count = 0
    for img in img_src.glob('*'):
        dst_img = f'{MERGED_DIR}/{split}/images/{prefix}_{img.name}'
        dst_lbl = f'{MERGED_DIR}/{split}/labels/{prefix}_{img.stem}.txt'
        shutil.copy(img, dst_img)
        lbl = lbl_src / f'{img.stem}.txt'
        if lbl.exists():
            shutil.copy(lbl, dst_lbl)
        count += 1
    return count

for split in ['train', 'valid', 'test']:
    c1 = merge_dataset(f'{DATA_DIR}/ds1', split, 'ds1')
    c2 = merge_dataset(f'{DATA_DIR}/ds2', split, 'ds2')
    c3 = merge_dataset(f'{DATA_DIR}/ds3', split, 'ds3')
    print(f'{split}: {c1 + c2 + c3} images merged')

## Step 4 — Create dataset.yaml

In [ ]:
# Unified defect classes
dataset_yaml = {
    'path': MERGED_DIR,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 5,
    'names': ['crack', 'corrosion', 'dent', 'surface_damage', 'fastener_damage']
}

yaml_path = f'{MERGED_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f)

print('dataset.yaml created:')
print(open(yaml_path).read())

## Step 5 — Train YOLOv8

In [ ]:
model = YOLO('yolov8m.pt')  # Medium model — best for accuracy on defects

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # Early stopping
    device=0,             # GPU
    project=f'{WORK_DIR}/runs',
    name='aeroinspect_v1',
    augment=True,         # Data augmentation
    mosaic=1.0,
    flipud=0.3,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

print('Training complete!')

## Step 6 — Evaluate Model

In [ ]:
metrics = model.val()
print(f'mAP50:     {metrics.box.map50:.3f}')
print(f'mAP50-95:  {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall:    {metrics.box.mr:.3f}')

## Step 7 — Export Model

In [ ]:
# Export to ONNX for production deployment
model.export(format='onnx', dynamic=True, simplify=True)

# Save best weights
best_weights = f'{WORK_DIR}/runs/aeroinspect_v1/weights/best.pt'
print(f'Best model saved at: {best_weights}')
print('Download best.pt and onnx model from Kaggle output panel')

## Step 8 — Test on Sample Image

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image

trained_model = YOLO(best_weights)

# Run inference on a test image
test_images = list(Path(f'{MERGED_DIR}/test/images').glob('*'))[:3]

for img_path in test_images:
    results = trained_model.predict(str(img_path), conf=0.3)
    result_img = results[0].plot()
    plt.figure(figsize=(10, 8))
    plt.imshow(result_img[:, :, ::-1])
    plt.title(f'Detections: {img_path.name}')
    plt.axis('off')
    plt.show()
    print(f'Boxes: {results[0].boxes}')